# Experiment 3 — Staging on RGB + Tissue-Probability Channels — Colab Run

Trains **MiT-B1 + CORN, ImageNet weights** (task_3's confirmed best config) on 8-channel samples: the full, uncropped RGB photo (channels 0-2) plus 5 per-pixel tissue-class probability channels (granulation, slough, necrosis, tendon, bone — channels 3-7) from Task 2 (MAnet+MiT-B2), zeroed where no wound was found. Same original PIID 3-fold split as task_3's own ablation (verified directly). config_id = `mit_b1__corn__exp3_tissue_channels`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Upload `wound-staging-masks-exp-3.zip` to `drive/MyDrive/thesis/` first (built locally by `more_experiments/exp/exp3_tissue_channels/scripts/generate_dataset.py`, ~144MB zipped, ~422MB extracted — 8-channel `.npy` arrays, not JPEGs).

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-3"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

Same `requirements.txt` as Task 3 — `scripts/train.py` reuses `task_3/src`'s engine/losses/metrics/splits/augmentations/utils directly; only `model.py` and `dataset.py` (8-channel versions) are local.

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp3_tissue_channels/scripts")
print(os.getcwd())

## 5. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-fold history or checkpoints.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp3_tissue_channels_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 6. Train — MiT-B1 + CORN, 8-channel input, 3-fold

Single config, no grid. No `--batch` override: uses `configs/base.yaml`'s `batch_size: 16`, matching task_3's own baseline ablation run exactly (an earlier exp-1/exp-2 run used `--batch 32` and had to be redone after the mismatch was caught — not repeating that here). `train.py` already skips completed `(config, fold)` rows and resumes a killed fold from its last saved epoch; this loop restarts the process on a crash. A full runtime disconnect kills the loop too — just re-run this cell, training picks up from the last saved epoch either way.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "train.py", "--workers", "4"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("training finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 7. Check results

3 rows expected (one per fold) once training completes.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())